Will import and play with the pdf science files present.

Testbed for importing pdf files.

Keep getting this message when testing:

LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``

In [51]:
# Imports that are not directly RAG
import os
import pandas
import pathlib
cwd = pathlib.Path.cwd()


In [52]:
import chromadb
# client = chromadb.PersistentClient(path="/path/to/save/to")
# can use client.reset() to delete the database

from langchain_classic.prompts import ChatPromptTemplate  # need to figure out how to get around this
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_community.llms.ollama import Ollama
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings



In [53]:
chroma_database_dir = cwd / "DB_of_Holding"
pdf_dir = cwd / "Folder_of_Holding"
# A test file. Any small pdf can be used, this is meant to be a test of the PDFPlumberLoader, not the content of the file itself.
test_file = cwd / pdf_dir / "BF01079761.pdf"  # this if an academic paper on Cost Disease in the Arts

# Another test file. This is meant to test vector embeddings, and it doesn't matter if it slips in with the rulebooks.
vector_test_file = cwd / pdf_dir /"Kerttu+Lehto+--+Role-Playing+Games+and+Well-Being+--+IJRP+11.pdf"  # this could be used for the basic test as well, it doesn't matter

In [54]:
class Open_Folder_of_Holding:
    '''
    This takes the files in the Folder of Holding and uploads them to the Chromoa Database.
    The folder of holding is hard coded to how this script works. It's just the way it's going to be.
    '''
    def __init__(self, chunk_size, chunk_overlap, *args, **kwargs) -> None:
        self.chunk_size, self.chunk_overlap = chunk_size, chunk_overlap



    def _create_chunks(self, document, *args, **kwargs):
        '''
        Chunks the document.
        '''
        split_text = RecursiveCharacterTextSplitter(chunk_size = self.chunk_size, chunk_overlap = self.chunk_overlap, add_start_index = True)

        chunks = split_text.split_documents(document)

        if not chunks:
             return None
        
        return chunks
    

    def _delete_database(self):
        '''
        This deletes the database. Useful for testing.

        This connects to the database via ChromaDB library itself.
        '''
        db = chromadb.PersistentClient(path = chroma_database_dir)
        db.reset()
    
    
    def _get_embeddings(self, *args, **kwargs):
        '''
        Returns the embedding model. Yes this is hard coded. This one probably has to be, because the embedding model has to be used to de-embed the information.
        '''
        return OllamaEmbeddings(model = "qwen3-embedding:4b")
    

    def _load2Chroma(self, chunks, *args, **kwargs):
        '''
        Loads the documents to the Chroma Database.

        Again, this folder is hard coded, and that's just the way it's going to be.

        To add to a Chroma Collection - which allows the data to be organized even better - you would use something like:
        
        client = chromadb.Client()
        collection = client.get_or_create_collection(name = "collection name")
        collection.add(documents = documents, ids = chunk_ids)

        The question becomes: how to do this dynamically without having to specify what the collection is by the user.

        This is going to be a long term addition. It looks like there are others looking and working on this too (https://github.com/zylon-ai/private-gpt/discussions/298)
        or maybe they already have a solution.
        '''
        db = Chroma(persist_directory = str(chroma_database_dir), embedding_function = self._get_embeddings())  # this line can take another kwarg, collction_name = "collection name", which will assign the documents to that collection.
                                                                                                                # how to do this dynamically and well organized?

        chunks = self._metadata_IDs(chunks)

        existing_items = db.get(include = [])
        existing_ids = set(existing_items["ids"])
        print(f"Number of existing documents in the Database: {len(existing_ids)}")

        new_chunks = []
        for chunk in chunks:
            if chunk.metadata["id"] not in existing_ids:
                new_chunks.append(chunk)

        if len(new_chunks):
            print(f"Adding {len(new_chunks)} new documents to database")
            new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
            db.add_documents(new_chunks, ids = new_chunk_ids)

        else:
            print("No new documents to add")



    def load_documents(self, *args, **kwargs):
        '''
        Loads the documents from the Folder of Holding. That folder will take everything that is in there, vectorize it, and see if it can be uploaded to the database
        '''
        document = None
        chunks = None
        for file in self._yield_documents():
            if file.suffix == ".pdf":
                document = self._load_pdf(file)

            elif file.suffix == ".txt":
                continue
            
            elif file.suffix == ".csv":
                continue

            else:
                continue
            
            if document is not None:
                chunks = self._create_chunks(document)

            if chunks is not None:
                self._load2Chroma(chunks)


    def _load_pdf(self, file, *args, **kwargs):
            '''
            This loads the pdf files vai PDF Plumber. PDF Plumber was chosen due to the number of tables that will be pulled: supposedly it's pretty good at that.
            '''
            loader = PDFPlumberLoader(file)
            document = loader.load()

            if not document:
                 return None
            
            return document


    def _metadata_IDs(self, chunks, *args, **kwargs):
        '''
        Assigns a new metadata ID to the item. The metadata tag is: source document: page: chunk index. The chunk index for each document goes from [0, max chunks].
        '''
        last_page_id = None
        current_chunk_index = 0

        for chunk in chunks:
            source = chunk.metadata.get("source")
            page = chunk.metadata.get("page")
            current_page_id = f"{source}:{page}"

            if current_page_id == last_page_id:
                current_chunk_index += 1
            else:
                current_chunk_index = 0

            chunk_id = f"{current_page_id}:{current_chunk_index}"
            last_page_id = current_page_id

            chunk.metadata["id"] = chunk_id

        return chunks
        

    def _vectorize(self, chunks, *args, **kwargs):
        '''
        Creates the vectorization of the data
        '''
        vectordb = Chroma.from_documents(documents = chunks, embedding = self._get_embeddings())
        return vectordb


    def _yield_documents(self, *args, **kwargs):
        '''
        This will yield the documents in the Folder of Holding. I'm using this to avoid using any legacy functions in langchain.
        '''    
        for root, _, files in os.walk(pdf_dir):
            for file in files:
                yield pathlib.Path(root, file)


Holding = Open_Folder_of_Holding(1000, 100)

In [55]:
class Search_Bundle_of_Holding:
    '''
    Storing some snippets of code here.
    
    This one gets all the documents in the database. Returns a dictionary that right (5/21/26) now has the following keys: 'ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'

    db = Chroma(persist_directory = str(chroma_database_dir))
    db_get:dict = db.get()
    '''
    def __init__(self) -> None:
        pass

        self.PROMPT_TEMPLATE = """
        Answer the question based only on the following context:

        {context}
        """


    def _get_embeddings(self, *args, **kwargs):
        '''
        Returns the embedding model.
        '''
        return OllamaEmbeddings(model = "qwen3-embedding:4b")
    

    def model_QnA(self, question, *args, **kwargs):
        '''
        '''
        model = Ollama(model = "phi4")
        
        results = self._query(question, *args, **kwargs)
        context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
        prompt_template = ChatPromptTemplate.from_template(self.PROMPT_TEMPLATE)
        prompt = prompt_template.format(context = context_text, question = question)

        response_text = model.invoke(prompt)

        sources = [doc.metadata.get("id", None) for doc, _score in results]
        formatted_response = f"Response: {response_text}\nSources: {sources}"
        print(formatted_response)

        return response_text
    

    def _query(self, question, k: int = 5, 
              *args, **kwargs):
        '''
        Finds the relevant information from the input.
        '''
        db = Chroma(persist_directory = str(chroma_database_dir), embedding_function=self._get_embeddings())

        results = db.similarity_search_with_score(question, k = k)

        # print(type(results))

        return results

Bundle = Search_Bundle_of_Holding()

In [56]:
# Holding.load_documents()


The challenge I'm having is that, to query, I have to know the collection, but I might not know the collection at all. So, the task is two fold:

1) assign data to collections
 
2) query all collections
 
3) query specific collections
 
5) learn to count

In [57]:
results = Bundle.model_QnA("what are roleplaying games?")

C:\Users\tokyo\AppData\Local\Temp\ipykernel_34696\3875919630.py:30: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  model = Ollama(model = "phi4")


Response: Based on the provided context:

1. **Role-Playing Games (RPGs):** According to Zagal and Deterding (2018b), role-playing involves players creating, enacting, and governing their characters' actions with significant freedom in choosing those actions. RPGs can be played in various formats: verbally using rulebooks and character sheets (tabletop RPGs), physically with one's own body (live-action role-play or larp), or through internet platforms (online RPGs).

2. **Origins and Research:** Role-playing as a cultural phenomenon is generally traced back to the publication of Dungeons & Dragons in 1974, although academic research on role-playing began earlier, in the late 1960s and 1970s, from education and sociology fields.

3. **Academic Discussions:** The early discussions about role-play focused on its importance and potential both for serious applications and entertainment (see Abt 1970). Gary Gygax and Dave Arneson, developers of Dungeons & Dragons, were involved in these acad